<p class="lead">A result that cannot be reproduced is not a weak result; it is not a result at all. This unit turns the project folder built in Unit 1 into an object that a stranger — a reviewer, a new student, you in three years — can rebuild from scratch and audit line by line. Four machineries do the work: <strong>version control</strong>, which records what the code was at every moment; <strong>environment pinning</strong>, which records what the software was; <strong>provenance</strong>, which records where each byte of data came from and proves it has not changed; and the <strong>FAIR principles</strong> with their licences and persistent identifiers, which make the whole thing findable and legally reusable. We do not narrate git — we compute its object identifiers ourselves in R, build a commit graph, merge two branches and produce a conflict, so that the commands stop being incantations.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad convierte una carpeta de proyecto en un objeto reproducible y auditable. Construimos el modelo de objetos de git calculando nosotros mismos los identificadores por contenido con <code>digest::digest()</code> (SHA-1 de <code>blob</code>, árbol y <em>commit</em>), armamos un grafo de commits, creamos una rama, la fusionamos y generamos un conflicto real; además intentamos ejecutar el <code>git</code> real con <code>system2()</code> y mostramos honestamente lo que ocurre en la máquina que compila esta página. Luego fijamos el entorno con <code>renv</code> y <code>sessionInfo()</code>, y mostramos por qué "funciona en mi máquina" es un problema de integridad de datos. Registramos la procedencia: un manifiesto con fuente, consulta, fecha, licencia y SHA-256 calculado en vivo sobre los archivos congelados del curso, verificado contra <code>manifest_rock1.json</code>. Medimos el efecto avalancha y el costo de verificar. Finalmente aplicamos los principios FAIR a un conjunto QSAR y a un modelo, revisamos licencias de código y de datos (incluida CC BY-SA de ChEMBL), el DOI de Zenodo y los archivos <code>README</code>, <code>LICENSE</code> y <code>CITATION.cff</code>, y terminamos con una auditoría mecánica de reproducibilidad.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain git's mental model — content-addressed snapshots in a directed acyclic graph — and compute an object identifier by hand in R.</li>
<li>Use git for daily work: init, add, commit, branch, merge, resolve a conflict; and GitHub for collaboration: fork, pull request, review, issues.</li>
<li>Write a <code>.gitignore</code> that keeps data and secrets out of the repository, and know where large files go instead.</li>
<li>Pin an environment with <code>renv</code> and <code>sessionInfo()</code>, and explain why "it works on my machine" is a data-integrity problem.</li>
<li>Record data provenance in a manifest — source, query, date, checksum, licence — and verify a file against it with <code>digest::digest(file = TRUE)</code>.</li>
<li>Apply the FAIR principles to a QSAR dataset and to a trained model, and score a real dataset against them.</li>
<li>Choose a licence for code and for data, deposit with a DOI, and write <code>README</code>, <code>LICENSE</code> and <code>CITATION.cff</code>.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Read it linearly with the code running. One chunk behaves differently depending on where you run it: the chunk that calls the real <code>git</code> executable through <code>system2()</code> works on your laptop and in Colab but may be refused by a locked-down build machine, and it prints whichever happened — the output you see on this page is the genuine output of the machine that built it. Everything else is pure R and runs anywhere. Beyond the Unit 1 stack this unit uses <code>digest</code> for checksums and <code>yaml</code> for the citation file; in a fresh Colab R runtime install them once with <code>install.packages(c("digest","yaml","data.table"))</code>. All temporary repositories and projects are created under <code>tempdir()</code> and deleted in the last chunk. This is the <strong>R edition</strong>; its <a href="../../Python/U2_Reproducibility/index.html">Python counterpart</a> performs the same experiments with <code>hashlib</code> and <code>subprocess</code>, and every number below was produced by the R code shown here.</p></div>



In [ ]:
## Setup: the stack used in this unit, plus the course data loader.
suppressPackageStartupMessages({
  library(data.table); library(dplyr); library(ggplot2)
  library(digest); library(jsonlite); library(yaml)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9.5) +
          theme(plot.title = element_text(face = "bold", size = 10.5),
                panel.grid.minor = element_blank()))

REPO <- "DanielBustosG/DanielBustosG.github.io"
DATA_URL <- sprintf("https://raw.githubusercontent.com/%s/main/DataScience/data/", REPO)
data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dir.create("data", showWarnings = FALSE); dest <- file.path("data", name)
  if (!file.exists(dest)) download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  dest
}
load_data <- function(name, ...) data.table::fread(data_path(name), ...)

SCRATCH <- normalizePath(file.path(tempdir(), "u2_scratch"), mustWork = FALSE)
dir.create(SCRATCH, recursive = TRUE, showWarnings = FALSE)

run <- function(args, dir = NULL) {
  ## Run an external command, returning (status, combined output). Never stops.
  out <- tryCatch(
    suppressWarnings(if (is.null(dir)) system2(args[1], args[-1], stdout = TRUE, stderr = TRUE)
                     else { old <- setwd(dir); on.exit(setwd(old)); 
                            system2(args[1], args[-1], stdout = TRUE, stderr = TRUE) }),
    error = function(e) structure(conditionMessage(e), status = 127L))
  st <- attr(out, "status"); if (is.null(st)) st <- 0L
  list(status = st, out = paste(out, collapse = "\n"))
}

cat(R.version.string, "|", R.version$platform, "\n")
for (p in c("data.table", "digest", "jsonlite", "yaml", "ggplot2"))
  cat(sprintf("%-11s %s\n", p, as.character(packageVersion(p))))
cat("git        ", run(c("git", "--version"))$out, "\n")
cat("scratch directory created\n")



## 1. The problem this unit solves

Computational drug discovery has a specific, well-documented reproducibility problem, and it is not caused by fraud or by incompetence. It is caused by four mundane facts. **Code is not released**, so a method described in prose cannot be re-run<sup class="c"><a href="#r1">[1]</a></sup>. **Databases move**: ChEMBL is re-released roughly twice a year, compounds are re-standardised, activities are added and occasionally corrected, so "data downloaded from ChEMBL" without a release number names no definite table<sup class="c"><a href="#r2">[2]</a></sup><sup class="c"><a href="#r3">[3]</a></sup>. **Randomness is unseeded**, so a reported metric is one draw from a distribution the paper does not show (we measure this in Section 3.3). And **curation is undocumented**: the choices that turn 5,089 raw activity records into a modelling table of a few thousand compounds — which units, which relation symbols, which duplicates, which outliers — change the result more than the choice of algorithm, and are usually described in a single sentence<sup class="c"><a href="#r4">[4]</a></sup>, even though the database's own curators document in detail how much judgement each of those steps involves<sup class="c"><a href="#r34">[34]</a></sup>.

The consequences are measurable across computational science: large surveys report that most researchers have failed to reproduce someone else's result<sup class="c"><a href="#r5">[5]</a></sup>, journal policies demanding data and code are honoured unevenly<sup class="c"><a href="#r6">[6]</a></sup>, artificial-intelligence research in particular has been diagnosed with a reproducibility crisis<sup class="c"><a href="#r7">[7]</a></sup><sup class="c"><a href="#r8">[8]</a></sup>, and high-profile models in biomedicine have been criticised precisely for missing code and data<sup class="c"><a href="#r9">[9]</a></sup>. The response from the community is not heroic: it is a short list of mundane practices<sup class="c"><a href="#r10">[10]</a></sup><sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r12">[12]</a></sup><sup class="c"><a href="#r32">[32]</a></sup>, assembled in the cross-disciplinary manifesto for reproducible science<sup class="c"><a href="#r30">[30]</a></sup> and in the community handbook this course follows<sup class="c"><a href="#r33">[33]</a></sup>. R has an unusually good position in that list, because the practice of writing the analysis and the report as one knitted document was argued for in this language first<sup class="c"><a href="#r35">[35]</a></sup>; what it does not give you for free is everything else in this unit.

<div class="box bridge"><h4>Domain context — why a "database version" is part of your data</h4>
<p>ChEMBL is a curated public database of bioactivity measurements extracted from the medicinal-chemistry literature<sup class="c"><a href="#r2">[2]</a></sup>. It is not a static file: each release adds newly published assays, revises compound structures after re-standardisation, and sometimes removes records flagged as erroneous. The course's frozen snapshot therefore records its <em>release</em> (ChEMBL_37), the exact query, and the download date — because a reader who repeats the same query against a later release will legitimately obtain a different table, and that difference must be attributable to the database rather than to you. The same holds for every data source in this laboratory: the Protein Data Bank re-processes depositions, UniProt re-annotates entries, and GEO series can be updated by their submitters. A dataset is identified by <em>source + query + date + checksum</em>, never by a file name.</p></div>

<div class="box key"><h4>Three words that are routinely confused</h4>
<p><strong>Repeatable</strong>: the same team, with the same code and data, gets the same answer. <strong>Reproducible</strong>: a different team, with <em>your</em> code and data, gets your answer. <strong>Replicable</strong>: a different team, with new data of the same kind, reaches the same conclusion. This unit buys you the first two; only science buys you the third. Confusing them is how a paper claims robustness it has not demonstrated<sup class="c"><a href="#r13">[13]</a></sup>.</p></div>



In [ ]:
## Conceptual figure: what each machinery of this unit protects.
layers <- data.frame(
  x = 1:5,
  name = c("CODE", "SOFTWARE", "DATA", "RANDOMNESS", "IDENTITY"),
  note = c("git: every version of every file,\nwith a message and an author",
           "renv.lock / sessionInfo():\nthe exact package versions",
           "manifest: source, query, date,\nSHA-256, licence",
           "explicit seeds, and results\nreported over many seeds",
           "DOI, licence, CITATION.cff:\nfindable, legally reusable"),
  fill = c("#dce8f5", "#f7e7d6", "#f6dcdc", "#dcefdc", "#ece7f6"), stringsAsFactors = FALSE)

p <- ggplot(layers) +
  geom_tile(aes(x, 0.5, fill = I(fill)), width = 0.88, height = 1.05, colour = "grey35",
            linewidth = 0.4) +
  geom_text(aes(x, 0.88, label = name), fontface = "bold", size = 2.9) +
  geom_text(aes(x, 0.37, label = note), size = 2.3) +
  geom_segment(aes(x = 0.55, xend = 5.45, y = -0.16, yend = -0.16), colour = "grey35",
               arrow = arrow(length = unit(0.08, "cm"), type = "closed")) +
  annotate("text", x = 3, y = -0.33, size = 2.8, colour = PAL[["orange"]],
           label = "a result is reproducible only when all five are recorded") +
  scale_x_continuous(limits = c(0.4, 5.6)) + scale_y_continuous(limits = c(-0.45, 1.2)) +
  labs(title = "The five things a reproducible result records") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 1.</b> The five records that together make a computational result reproducible, and the artefact that carries each one. A project missing any one of them fails in a characteristic way: without the code version you cannot tell which analysis produced the figure; without the environment the code may run and give different numbers; without the data manifest you cannot tell which download was used; without seeds you cannot reproduce a single run; without a licence and an identifier nobody may legally reuse what you published. Sections 2 to 6 build them in that order.</p>

## 2. git, computed rather than narrated

Most introductions to git teach commands. The commands are memorable only once the model underneath them is clear, and that model is unusually simple: **git stores snapshots of whole files, addressed by the hash of their contents, and links them into a directed acyclic graph (DAG) of commits.** There are exactly four object types — *blob* (the contents of a file), *tree* (a directory listing: names pointing at blobs and other trees), *commit* (a tree plus parent commits plus a message and an author), and *tag* (a name for a commit). A *branch* is nothing but a movable pointer to a commit.

### 2.1 A worked example by hand: the object identifier

Git names an object by the SHA-1 hash of a short header followed by the content. For a file containing exactly the twelve bytes `hello world` plus a newline, the hashed byte string is

\[ \texttt{"blob"} \;+\; \texttt{" "} \;+\; \texttt{"12"} \;+\; \texttt{NUL} \;+\; \texttt{"hello world\\n"} . \]

Counting the content by hand: `hello world` is 11 characters (5 + 1 + 5) and the trailing newline makes **12**, which is the number that goes in the header. The separator between header and content is a single zero byte, written `as.raw(0)` in R and `NUL` above. So the hashed input is the 20-byte sequence `blob 12<NUL>hello world<LF>`, and nothing else about the file — not its name, not its timestamp, not its permissions — enters the identifier. Two files with the same bytes in two different repositories therefore have the same object id, which is why git can deduplicate and verify at the same time.

In R the hash of a *byte sequence* must be requested explicitly: `digest()` by default hashes the internal serialisation of an R object, which is not the same thing at all. Passing `serialize = FALSE` to a raw vector hashes exactly those bytes, and that distinction is the single most common mistake when reproducing a published checksum in R.



In [ ]:
## The worked example: build git's hashed byte string by hand, then ask git itself.
content <- charToRaw("hello world\n")
header  <- charToRaw(paste0("blob ", length(content)))
payload <- c(header, as.raw(0), content)
by_hand <- digest(payload, algo = "sha1", serialize = FALSE)

cat("content length (bytes):", length(content), "\n")
cat("hashed input (hex)    :", paste(as.character(payload), collapse = " "), "\n")
cat("SHA-1 computed here   :", by_hand, "\n")

tmpf <- file.path(SCRATCH, "hello.txt")
writeBin(content, tmpf)
g <- run(c("git", "hash-object", tmpf))
cat("git hash-object says  :", if (g$status == 0) g$out else paste0("[git unavailable: ", g$out, "]"), "\n")
cat("the two agree         :", if (g$status == 0) identical(trimws(g$out), by_hand) else "(not checked)", "\n")

## Changing one byte changes the identifier completely; the NAME changes nothing.
alt <- digest(c(header, as.raw(0), charToRaw("hello worlD\n")), algo = "sha1", serialize = FALSE)
common <- 0L
for (i in seq_len(40)) if (substr(by_hand, i, i) == substr(alt, i, i)) common <- i else break
cat("\nsame file, one capital letter:", alt, "\n")
cat("shared leading hex characters:", common, "\n")
cat("digest() WITHOUT serialize = FALSE hashes the R object, not the bytes:",
    digest(payload, algo = "sha1"), "\n")



The hand-built byte string and the hash produced by the real program agree (when `git` is available on the machine that rendered this page; the chunk prints which case applied). Three consequences follow and all three are used daily. Because the identifier is a function of content alone, **git cannot lose track of what a file contained** — if the bytes are recoverable, the id proves which version they are. Because changing a single letter changes essentially all of the hexadecimal digits, as the shared-prefix count shows, an identifier is a fingerprint rather than a summary: you cannot infer "nearly the same file" from "nearly the same id". And the last printed line is the R-specific warning: the same `digest()` call without `serialize = FALSE` returns a completely different hash, because it hashed R's serialisation of the raw vector rather than the twenty bytes themselves.

### 2.2 A miniature git

With a single content-addressing primitive, the rest of the model fits in one chunk. The implementation below is a faithful miniature: blobs, trees, commits and branch pointers, with identifiers computed exactly as git computes them. The one deliberate simplification is that our commits carry no timestamp, so the ids are reproducible across runs of this document; real git commit ids differ between runs for exactly that reason, which is itself worth knowing.



In [ ]:
## A faithful miniature of git's object store: blobs, trees, commits, branches.
OBJECTS <- new.env(parent = emptyenv())

store <- function(kind, payload_chr) {
  ## Content-address a payload exactly as git does: sha1(kind + " " + len + NUL + payload).
  body <- charToRaw(payload_chr)
  full <- c(charToRaw(paste0(kind, " ", length(body))), as.raw(0), body)
  oid <- digest(full, algo = "sha1", serialize = FALSE)
  assign(oid, list(kind = kind, payload = payload_chr), envir = OBJECTS)
  oid
}
write_blob <- function(text) store("blob", text)
write_tree <- function(entries) {                 # entries: named character vector name -> oid
  entries <- entries[order(names(entries))]
  store("tree", paste(sprintf("blob %s\t%s", entries, names(entries)), collapse = "\n"))
}
write_commit <- function(tree, parents, message, author = "student@lbqc") {
  store("commit", paste0("tree ", tree, "\n",
                         paste0(sprintf("parent %s\n", parents), collapse = ""),
                         "author ", author, "\n\n", message, "\n"))
}

b_readme <- write_blob("# ROCK1 activity project\n")
b_src1   <- write_blob("load <- function(path) data.table::fread(path)\n")
t1 <- write_tree(c("README.md" = b_readme, "R/load.R" = b_src1))
c1 <- write_commit(t1, character(0), "init: project skeleton and loader")

b_curate <- write_blob("curate <- function(df) df[relation == '=']\n")
t2 <- write_tree(c("README.md" = b_readme, "R/load.R" = b_src1, "R/curate.R" = b_curate))
c2 <- write_commit(t2, c1, "feat: curation step, exact relations only")

cat("objects stored so far:", length(ls(OBJECTS)), "\n")
cat("README blob reused in both commits:", substr(t1, 1, 8), "and", substr(t2, 1, 8),
    "both point at", substr(b_readme, 1, 8), "\n")
for (oid in ls(OBJECTS)[1:4]) {
  o <- get(oid, envir = OBJECTS)
  cat(sprintf("  %s  %-6s %4d bytes  %s\n", substr(oid, 1, 10), o$kind, nchar(o$payload),
              substr(gsub("\n", "\\\\n", o$payload), 1, 44)))
}
cat("\ncommit 2 payload:\n", get(c2, envir = OBJECTS)$payload, sep = "")



Read the printed commit payload: a commit is a tiny text object naming one tree, zero or more parents, an author and a message. Note what did *not* happen when `README.md` was unchanged between the two commits — no second copy was stored. Both trees point at the same blob id, because the content is the same. This is why a repository with a thousand commits of a project is not a thousand copies of the project, and why committing often is cheap.

### 2.3 Branching, merging and a real conflict

A **branch** is a name pointing at a commit. Creating one costs 41 bytes. Two people (or one person on two questions) commit on two branches; merging them produces a commit with *two* parents, and git computes the merged content by comparing both sides against their **merge base**, the most recent common ancestor. When both sides changed the same line, git cannot choose and writes a **conflict** into the file for a human to resolve.



In [ ]:
## Two branches, a merge commit, and a three-way merge that produces a real conflict.
BRANCHES <- c(main = c2, `feature/log-scale` = c2)      # both start from commit 2

base_text <- c("THRESHOLD <- 1000", "curate <- function(df) {", "  df[relation == '=']", "}")
ours_text <- c("THRESHOLD <- 1000", "curate <- function(df) {", "  df <- df[units == 'nM']",
               "  df[relation == '=']", "}")
theirs_text <- c("THRESHOLD <- 500", "curate <- function(df) {", "  df[relation == '=']", "}")

three_way_merge <- function(base, ours, theirs) {
  ## Line-based three-way merge, as git does it: take the side that changed; flag both-changed.
  n <- max(length(base), length(ours), length(theirs))
  pad <- function(x) c(x, rep(NA_character_, n - length(x)))
  b <- pad(base); o <- pad(ours); t <- pad(theirs)
  merged <- character(0); conflicts <- 0L
  for (i in seq_len(n)) {
    if (identical(o[i], t[i])) line <- o[i]
    else if (identical(o[i], b[i])) line <- t[i]
    else if (identical(t[i], b[i])) line <- o[i]
    else {
      conflicts <- conflicts + 1L
      merged <- c(merged, "<<<<<<< ours", if (is.na(o[i])) "" else o[i], "=======",
                  if (is.na(t[i])) "" else t[i], ">>>>>>> theirs")
      next
    }
    if (!is.na(line)) merged <- c(merged, line)
  }
  list(text = merged, conflicts = conflicts)
}

mg <- three_way_merge(base_text, ours_text, theirs_text)
cat("conflicting hunks:", mg$conflicts, "\n")
cat(paste(mg$text, collapse = "\n"), "\n")

## After the human resolves the conflict, the merge commit has TWO parents.
resolved <- mg$text[!grepl("^(<<<<<<<|=======|>>>>>>>|THRESHOLD)", mg$text)]
resolved <- c("THRESHOLD <- 500", resolved)
t3 <- write_tree(c("README.md" = b_readme, "R/load.R" = b_src1,
                   "R/curate.R" = write_blob(paste(resolved, collapse = "\n"))))
c_main <- write_commit(write_tree(c("README.md" = b_readme,
                                    "R/curate.R" = write_blob(paste(ours_text, collapse = "\n")))),
                       c2, "fix: keep only nanomolar records")
c_feature <- write_commit(write_tree(c("README.md" = b_readme,
                                       "R/curate.R" = write_blob(paste(theirs_text, collapse = "\n")))),
                          c2, "feat: lower the activity threshold to 500 nM")
c_merge <- write_commit(t3, c(c_main, c_feature), "merge feature/log-scale into main")
BRANCHES["main"] <- c_merge

parents <- regmatches(get(c_merge, envir = OBJECTS)$payload,
                      gregexpr("(?<=parent )[0-9a-f]+", get(c_merge, envir = OBJECTS)$payload,
                               perl = TRUE))[[1]]
cat("\nmerge commit", substr(c_merge, 1, 10), "has parents:",
    paste(substr(parents, 1, 10), collapse = ", "), "\n")
cat("conflict markers remaining after resolution:", sum(grepl("<<<<<<<", resolved)), "\n")



The printed file is what a conflict actually looks like: the two incompatible versions of the threshold line, wrapped in markers, with the lines that only one side touched merged silently. Note that the `units == 'nM'` filter added on `main` was merged **without a conflict**, because the other side did not touch that region — most merges are automatic, and the manual work is confined to genuinely contradictory edits. Note also the merge commit's two parents: this is why the history is a graph and not a list, and why `git log --graph` draws what it draws.

<div class="box warn"><h4>A conflict is not an error</h4>
<p>Students routinely panic, delete the branch and re-clone. A conflict means git refused to guess between two intentional edits — exactly the behaviour you want from a system holding your thesis. Resolve it by editing the file (removing the markers), <code>git add</code> it, and <code>git commit</code>. The one genuinely destructive mistake in this area is <code>git push --force</code> on a shared branch, which rewrites history other people have built on. A second, R-specific hazard: knitted <code>.html</code> outputs and <code>.RData</code> files are binary or near-binary, so git cannot merge them at all and every collaboration produces a conflict in them. Commit the <code>.Rmd</code> source and regenerate the output, or add the output to <code>.gitignore</code>.</p></div>



In [ ]:
## The commit graph of the miniature repository, drawn from its own parent links.
nodes <- data.frame(
  oid = c(c1, c2, c_main, c_feature, c_merge),
  x = c(0, 1, 2, 2, 3), y = c(0, 0, 0.55, -0.55, 0),
  label = c("init", "curation step", "nM filter (main)", "threshold 500 (feature)", "merge"),
  stringsAsFactors = FALSE)
edges <- do.call(rbind, lapply(seq_len(nrow(nodes)), function(i) {
  pay <- get(nodes$oid[i], envir = OBJECTS)$payload
  ps <- regmatches(pay, gregexpr("(?<=parent )[0-9a-f]+", pay, perl = TRUE))[[1]]
  ps <- ps[ps %in% nodes$oid]
  if (!length(ps)) return(NULL)
  data.frame(x = nodes$x[match(ps, nodes$oid)] + 0.12, y = nodes$y[match(ps, nodes$oid)],
             xend = nodes$x[i] - 0.12, yend = nodes$y[i])
}))

p <- ggplot() +
  geom_segment(data = edges, aes(x = xend, y = yend, xend = x, yend = y), colour = "grey55",
               arrow = arrow(length = unit(0.08, "cm"), type = "closed")) +
  geom_point(data = nodes, aes(x, y), size = 11, shape = 21, fill = "#dce8f5",
             colour = PAL[["blue"]], stroke = 0.7) +
  geom_text(data = nodes, aes(x, y, label = substr(oid, 1, 4)), size = 2.4, family = "mono") +
  geom_text(data = nodes, aes(x, y + ifelse(y <= 0, -0.3, 0.3), label = label), size = 2.5) +
  annotate("text", x = 3.0, y = 0.42, label = "main ->", hjust = 1, size = 2.8,
           colour = PAL[["orange"]], fontface = "bold") +
  scale_x_continuous(limits = c(-0.4, 3.8)) + scale_y_continuous(limits = c(-1.0, 0.9)) +
  labs(title = "The commit DAG built in this document (arrows point from child to parent)") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)

kinds <- vapply(ls(OBJECTS), function(o) get(o, envir = OBJECTS)$kind, character(1))
cat("branch pointers:", paste(names(BRANCHES), substr(BRANCHES, 1, 8), sep = " -> ",
                              collapse = " | "), "\n")
cat("total objects:", length(kinds), "| blobs:", sum(kinds == "blob"),
    "| trees:", sum(kinds == "tree"), "| commits:", sum(kinds == "commit"), "\n")



<p class="caption"><b>Figure 2.</b> The commit graph produced by the two previous chunks, drawn by following each commit's own <code>parent</code> lines. Each node is labelled with the first four hexadecimal digits of its real content-addressed identifier. The graph is a DAG, not a chain: two commits share the parent <code>curation step</code>, and the merge commit has two parents. The <code>main</code> branch is a pointer at the rightmost node — moving it is all that <code>git merge</code> does once the content is computed. The object census printed below the figure shows how few objects a whole project history needs.</p>

### 2.4 The daily commands, and what each one does to the graph

| command | what it does to the model |
|---|---|
| `git init` | creates the object store and an empty `main` pointer |
| `git add <f>` | writes a blob for `<f>` and records it in the staging area (the next tree) |
| `git commit -m "…"` | writes a tree from the staging area and a commit pointing at it and at the current branch tip; moves the pointer |
| `git log --oneline --graph` | walks the DAG backwards from the branch pointer |
| `git branch <name>` | creates a new pointer at the current commit |
| `git switch <name>` | moves `HEAD` to another pointer and rewrites the working files to its tree |
| `git merge <name>` | computes the merge base, merges the two trees, writes a commit with two parents |
| `git remote add origin <url>` / `git push` | copies objects the remote lacks and moves its pointer |
| `git status` / `git diff` | compares working files, staging area and current commit |
| `git revert <commit>` | writes a *new* commit undoing a change (safe); `git reset --hard` rewrites history (dangerous) |

Commit messages are part of the scientific record. The convention used in this group is the conventional-commit form — `feat:`, `fix:`, `docs:`, `data:`, `refactor:`, `test:` followed by an imperative summary — because it makes `git log` readable as a change log, and because "fixed stuff" tells a reviewer nothing. Collaboration adds four GitHub operations to the same model: **fork** (your own copy of someone's repository), **branch**, **pull request** (a request to merge your branch, with review and discussion attached), and **issue** (a unit of work with a number that commit messages can reference). The published guidance on using git and GitHub in research<sup class="c"><a href="#r14">[14]</a></sup><sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r16">[16]</a></sup> amounts to the workflow this group follows: one branch per question, one pull request per branch, review before merge, and the `main` branch always runnable.



In [ ]:
## Does the real git run here? Attempt a repository under tempdir() and report honestly.
demo_repo <- file.path(SCRATCH, "demo_repo")
dir.create(demo_repo, recursive = TRUE, showWarnings = FALSE)
init <- run(c("git", "init", "-q", "."), dir = demo_repo)
msg <- gsub(SCRATCH, "<scratch>", strsplit(init$out, "\n")[[1]][1], fixed = TRUE)
cat("$ git init      ->", if (init$status == 0) "ok" else paste("refused:", msg), "\n")

if (init$status == 0) {
  writeLines("# demo", file.path(demo_repo, "README.md"))
  cfg <- c("-c", "user.name=LBQC student", "-c", "user.email=student@example.org")
  run(c("git", "add", "README.md"), dir = demo_repo)
  run(c("git", cfg, "commit", "-q", "-m", "docs: first commit"), dir = demo_repo)
  dir.create(file.path(demo_repo, "R"), showWarnings = FALSE)
  writeLines("THRESHOLD <- 1000", file.path(demo_repo, "R", "curate.R"))
  run(c("git", "add", "R/curate.R"), dir = demo_repo)
  run(c("git", cfg, "commit", "-q", "-m", "feat: curation threshold"), dir = demo_repo)
  cat("\n$ git log --oneline --graph\n",
      run(c("git", "log", "--oneline", "--graph"), dir = demo_repo)$out, "\n", sep = "")
  cat("\n$ git cat-file -p HEAD\n",
      run(c("git", "cat-file", "-p", "HEAD"), dir = demo_repo)$out, "\n", sep = "")
} else {
  cat("\nThis build machine forbids creating repositories (a sandbox restriction, not a git error),\n")
  cat("so the transcript above is the refusal itself. Run this chunk on your laptop or in Colab\n")
  cat("to see the real log; the object model demonstrated in Sections 2.1-2.3 is computed\n")
  cat("locally in R and is identical to what git would store.\n")
}



Whatever the chunk printed is the truth about the machine that rendered this page. On a normal workstation or in Colab you see a two-commit graph and the raw text of a commit object — the same four fields our miniature wrote in Section 2.2. On the locked-down machine used to build the course pages, repository creation is refused by the sandbox, and the chunk says so rather than pretending. This is itself a small lesson in provenance: **an environment that silently substitutes one behaviour for another is worse than one that refuses loudly**.

### 2.5 What must never enter a repository

Two kinds of file do not belong in git: **data** (because git stores every version forever, and a 100 MB table committed ten times is a 1 GB clone) and **secrets** (because a key pushed once is public forever, even if the next commit removes it). Data goes to a data repository with a checksum and a DOI (Section 6), to an institutional store, or — if it must be versioned alongside the code — to git-lfs or a data-version-control tool that keeps pointers in git and bytes elsewhere. Secrets go in an untracked `.Renviron` file read by `Sys.getenv()`, or the operating system's keyring. The `.gitignore` file states the rules, and the next chunk applies the group's standard rules to the real course data folder, using R's `glob2rx()` to turn each glob into the regular expression that implements it.



In [ ]:
## Apply a .gitignore to the real data folder: which files would be committed, and which ignored?
GITIGNORE <- c("data/raw/*", "data/processed/*", "!*.gitkeep", "*.csv", "*.gz", "*.npz",
               "*.xtc", "*.sdf", ".Rhistory", ".RData", "*.Renviron", "*.key")

ignored <- function(relpath, rules) {
  ## Minimal .gitignore semantics: last matching rule wins; '!' re-includes.
  decision <- FALSE
  for (rule in rules) {
    negate <- startsWith(rule, "!")
    pattern <- sub("^!", "", rule)
    rx <- glob2rx(basename(pattern))
    if (grepl(rx, basename(relpath)) || grepl(glob2rx(pattern), relpath)) decision <- !negate
  }
  decision
}

files <- sort(list.files(file.path("..", "data")))
tbl <- data.table(file = files,
                  MB = round(file.size(file.path("..", "data", files)) / 1e6, 3),
                  ignored = vapply(files, ignored, logical(1), rules = GITIGNORE))
print(knitr::kable(tbl[order(-MB)][1:14], format = "simple"))
cat(sprintf("\nwould be committed: %d files, %.2f MB | ignored: %d files, %.2f MB\n",
            sum(!tbl$ignored), sum(tbl$MB[!tbl$ignored]),
            sum(tbl$ignored), sum(tbl$MB[tbl$ignored])))
cat("committable files:", paste(tbl$file[!tbl$ignored], collapse = ", "), "\n")



<p class="caption"><b>Table (Section 2.5).</b> The fourteen largest files in the course's <code>data/</code> folder, their size, and whether the group's standard <code>.gitignore</code> rules would keep them out of the repository; the totals below cover all of them. The totals are the point: the ignored bytes are the ones that would otherwise be stored forever, in every clone, for every version. What remains committable are the small text artefacts that <em>describe</em> the data — the manifests, the READMEs and the scripts that built it — which is exactly the right division of labour.</p>

<div class="exercise"><h4>Exercise 1</h4>
<p>Add the rule <code>!manifest_*.json</code> to <code>GITIGNORE</code> and re-run the chunk. Does the decision for <code>manifest_rock1.json</code> change, and why not — which earlier rule already allowed it? Now add <code>*.json</code> before it and re-run: explain the difference in terms of "last matching rule wins". Then write the rule that would ignore every <code>.csv</code> in the folder <em>except</em> <code>rock1_curation_log.csv</code>, and confirm it with the function. Finally, note what our 12-line implementation does <em>not</em> do that real git does — directory-anchored patterns, <code>**</code>, and per-directory <code>.gitignore</code> files — and say why a teaching implementation is still worth writing.</p></div>

## 3. Pinning the environment

An environment specification answers the question "what software produced this number?". Three levels of strictness exist, and they serve different purposes.

- A **minimal specification** names the packages a project needs, with loose or no versions — in R, the `Imports:` field of a `DESCRIPTION` file, or a hand-written list at the top of a script. It is readable, it is what you maintain by hand, and it does *not* reproduce an environment: it describes an intention.
- A **full export** names every installed package with its exact version, including everything pulled in as a dependency. In R this is `sessionInfo()` (what was loaded) or `installed.packages()` (what was available). It documents an environment on a similar machine, and it is unreadable by humans.
- A **lock file** records exact versions *and* their sources, so the installation can be rebuilt and verified. In R this is `renv.lock`, written by `renv::snapshot()` and consumed by `renv::restore()`; it is what a continuous-integration system and a container image should install from.

The rule in this group: commit a readable minimal specification *and* the lock file; regenerate the lock file whenever the environment changes; and print `sessionInfo()` at the end of every analysis document, because that is the record of what actually ran. The next chunk captures the environment of this very document in all three forms. `renv` is not installed on the build machine, so the lock file is written here by hand in exactly the schema `renv::snapshot()` produces — the fields are the point, not the function call.



In [ ]:
## Capture the environment this document is running in, in three forms.
CORE <- c("data.table", "dplyr", "ggplot2", "digest", "jsonlite", "yaml")
versions <- vapply(CORE, function(p) as.character(packageVersion(p)), character(1))

description <- c("Package: rock1", "Type: Project", "Title: ROCK1 activity project",
                 paste0("Depends: R (>= ", paste(R.version$major, R.version$minor, sep = "."), ")"),
                 paste0("Imports: ", paste(CORE, collapse = ", ")))
writeLines(description, file.path(SCRATCH, "DESCRIPTION"))

lock <- list(R = list(Version = paste(R.version$major, R.version$minor, sep = "."),
                      Repositories = list(list(Name = "CRAN", URL = "https://cloud.r-project.org"))),
             Packages = setNames(lapply(CORE, function(p)
               list(Package = p, Version = versions[[p]], Source = "Repository",
                    Repository = "CRAN")), CORE))
write_json(lock, file.path(SCRATCH, "renv.lock"), auto_unbox = TRUE, pretty = TRUE)

cat("DESCRIPTION (minimal specification, maintained by hand):\n\n")
cat(paste(description, collapse = "\n"), "\n\n")
cat("renv.lock (exact versions, generated):\n\n")
cat(substr(paste(readLines(file.path(SCRATCH, "renv.lock")), collapse = "\n"), 1, 520), "...\n\n")
si <- sessionInfo()
cat("sessionInfo() essentials:", si$R.version$version.string, "|", si$running, "\n")
cat("attached packages:", length(si$otherPkgs), "| loaded via a namespace:", length(si$loadedOnly), "\n")
cat("BLAS behind the numerics:", basename(si$BLAS), "| LAPACK:", basename(si$LAPACK), "\n")



All three artefacts are now on disk and all three are needed. The `DESCRIPTION` is the one a human reads and edits; the lock file is the one that rebuilds the library; `sessionInfo()` is the one that proves what was actually attached when the document ran. Note what the lock file does *not* capture, and what the last two printed lines expose: the number of packages loaded only via a namespace is larger than the number attached (Section 3.1 quantifies this), and the BLAS and LAPACK libraries behind every matrix operation are system components, not R packages — which is why bit-identical reproduction of floating-point results across machines is not guaranteed by a lock file alone, and why containers exist<sup class="c"><a href="#r17">[17]</a></sup><sup class="c"><a href="#r18">[18]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> (Units 24 and 25 treat them properly).

### 3.1 How much software is actually behind six lines

A minimal specification of six packages is not a small dependency graph. The next chunk walks the real installed metadata to count the transitive closure — the packages that would be installed, and therefore the packages whose versions could change your numbers.



In [ ]:
## How many packages does each 'direct dependency' really pull in? Walk the installed metadata.
ip <- installed.packages()
BASE <- rownames(ip)[!is.na(ip[, "Priority"]) & ip[, "Priority"] %in% c("base", "recommended")]

deps_of <- function(pkg) {
  if (!pkg %in% rownames(ip)) return(character(0))
  fields <- ip[pkg, c("Depends", "Imports", "LinkingTo")]
  parts <- unlist(strsplit(fields[!is.na(fields)], ","))
  parts <- trimws(sub("\\(.*", "", parts))
  setdiff(parts[nzchar(parts)], c("R", BASE))
}
closure <- function(pkg, seen = character(0), depth = 0) {
  if (pkg %in% seen || depth > 8) return(seen)
  seen <- c(seen, pkg)
  for (d in deps_of(pkg)) seen <- closure(d, seen, depth + 1)
  seen
}

counts <- vapply(CORE, function(p) length(closure(p)) - 1L, integer(1))
total <- length(unique(unlist(lapply(CORE, closure))))
cat("transitive dependencies per direct requirement (base/recommended excluded):\n")
print(counts)
cat("distinct packages in the union of all closures:", total, "\n")

d <- data.frame(pkg = factor(names(counts), levels = names(counts)), n = as.integer(counts))
p <- ggplot(d, aes(pkg, n)) +
  geom_col(fill = PAL[["blue"]], width = 0.6) +
  geom_text(aes(label = n), vjust = -0.4, size = 3, fontface = "bold") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.14))) +
  labs(x = NULL, y = "transitive dependencies",
       title = sprintf("Six declared packages, %d packages in the closure", total))
print(p)



<p class="caption"><b>Figure 3.</b> The number of transitive dependencies each declared package brings into the environment, computed from the installed metadata of the machine that built this page, with the size of the union printed in the title. Base and recommended packages — which ship with R itself and are therefore pinned by the R version — are excluded, so the counts are the packages a lock file must actually record. The declared list is short; the real software surface is several times larger, and every package in it has a version that can change behaviour. This is the quantitative argument for recording a lock file rather than only a hand-written list, and for not adding a dependency casually.</p>

### 3.2 Why "it works on my machine" is a data-integrity problem

The phrase sounds like a communication failure. It is not: if a number in your thesis depends on an unrecorded package version, that number is a property of a laptop rather than of the data and the method. The failure is of the same kind as losing a notebook page in a wet-lab experiment. Published guidance for machine learning in the life sciences makes the recording of the computational environment a minimum standard, alongside data and code<sup class="c"><a href="#r11">[11]</a></sup><sup class="c"><a href="#r20">[20]</a></sup>.

### 3.3 The seed lottery, measured on a real model

Unseeded randomness is the most quietly damaging of the four causes listed in Section 1, because it produces a *plausible* number every time. The next chunk fits the same model — an ordinary least-squares regression of pActivity on nine descriptors — on the course's curated ROCK1 table, changing only the random split, and records what the reported test performance would have been.



In [ ]:
## The same model, 200 different random splits: what does 'the' R-squared mean?
cur <- load_data("rock1_chembl_curated.csv")
exact <- cur[censored == FALSE & !is.na(pActivity)]
FEATURES <- c("MolWt", "MolLogP", "TPSA", "HBD", "HBA", "RotB", "AromaticRings",
              "FractionCSP3", "QED")
dat <- as.data.frame(exact[, c(FEATURES, "pActivity"), with = FALSE])
cat("compounds with an exact pActivity:", nrow(dat), "| descriptors:", length(FEATURES), "\n")

scores <- numeric(200)
for (seed in seq_len(200)) {
  set.seed(seed)                                   # only the split changes; the model is identical
  test_i <- sample(nrow(dat), size = round(0.25 * nrow(dat)))
  fit <- lm(pActivity ~ ., data = dat[-test_i, ])
  pred <- predict(fit, newdata = dat[test_i, ])
  y <- dat$pActivity[test_i]
  scores[seed] <- 1 - sum((y - pred)^2) / sum((y - mean(y))^2)   # out-of-sample R^2
}
cat(sprintf("test R^2 over 200 seeds: min %.3f | median %.3f | max %.3f | sd %.3f\n",
            min(scores), median(scores), max(scores), sd(scores)))
cat(sprintf("a researcher reporting their luckiest seed would claim %.3f; the honest median is %.3f\n",
            max(scores), median(scores)))
cat(sprintf("relative inflation of the best seed over the median: %.0f%%\n",
            100 * (max(scores) / median(scores) - 1)))


In [ ]:
## Figure: the distribution of a 'single number' that depends only on the split.
marks <- data.frame(value = c(median(scores), max(scores), min(scores)),
                    label = c("median", "luckiest seed", "unluckiest seed"),
                    colour = c("black", PAL[["orange"]], PAL[["green"]]), stringsAsFactors = FALSE)
p <- ggplot(data.frame(r2 = scores), aes(r2)) +
  geom_histogram(bins = 28, fill = "#dce8f5", colour = PAL[["blue"]], linewidth = 0.3) +
  geom_vline(data = marks, aes(xintercept = value), colour = marks$colour,
             linetype = "dashed", linewidth = 0.5) +
  geom_text(data = marks, aes(x = value, y = Inf, label = sprintf(" %s\n %.3f", label, value)),
            colour = marks$colour, hjust = 0, vjust = 1.3, size = 2.5) +
  labs(x = expression("test " * R^2 * " of the same model, as a function of the random split only"),
       y = "number of seeds", title = "The seed lottery: 200 splits of one dataset and one model")
print(p)



<p class="caption"><b>Figure 4.</b> Distribution of the test \( R^2 \) obtained by refitting one fixed model on one fixed dataset, changing nothing but the random train/test split, over 200 seeds. The spread between the unluckiest and the luckiest seed is a substantial fraction of the median value itself (the exact numbers are printed by the previous chunk). A paper reporting a single unseeded split reports one draw from this histogram, and a paper that tried several splits and reported the best reports its right tail. The defence is not a better seed: it is reporting the distribution — repeated cross-validation with fixed seeds, as Unit 12 does properly.</p>

<div class="box key"><h4>What to record about randomness</h4>
<p>Record the seed, the number of repetitions, and the <em>dispersion</em> of the metric, not only its central value. If a result survives only for one seed, it is not a result. In this group a reported performance figure always carries either a standard deviation over repeated splits or a confidence interval from the bootstrap (Unit 7). Two R-specific details belong here. First, R's stream is global, so the position of <code>set.seed()</code> relative to every sampling call is part of the analysis — the loop above seeds <em>inside</em> the iteration precisely so that the 200 results are a function of the seed index and nothing else. Second, R's default sampling algorithm changed in version 3.6.0; a script written earlier reproduces its old numbers only under <code>RNGkind(sample.kind = "Rounding")</code>, which is why the R version belongs in the lock file. The seed lottery is the mildest of the validation failures catalogued across machine-learning-based science; the severe one is <em>leakage</em>, in which information from the test set reaches the model and inflates the metric systematically rather than randomly<sup class="c"><a href="#r31">[31]</a></sup> — Units 11 and 12 are built around preventing it.</p></div>

## 4. Provenance: proving a file is the file

A **manifest** is a small text file recording, for every data file in the project: where it came from, by which query, on which date, under which licence, and its cryptographic checksum. The checksum is what makes the record verifiable rather than merely declarative. A **cryptographic hash function** maps any number of bytes to a fixed-length digest such that changing a single bit changes the digest unpredictably, and finding two inputs with the same digest is computationally infeasible. SHA-256 is the current default; git uses SHA-1 for object naming, which is adequate for integrity but no longer for adversarial collision resistance.

The course's own data folder carries such a manifest. The next chunk recomputes the checksum of each frozen ROCK1 file from the bytes on disk and compares it with the recorded value — the operation a reviewer performs when they want to know whether the file they downloaded is the file you described. In R this is `digest::digest(path, algo = "sha256", file = TRUE)`, which streams the file rather than reading it whole; note again that **without `file = TRUE` the call would hash the file *name*** and return a confident, meaningless answer.



In [ ]:
## Verify the course's own frozen dataset against its manifest.
sha256_of <- function(path) digest(path, algo = "sha256", file = TRUE)

manifest <- fromJSON(data_path("manifest_rock1.json"), simplifyVector = FALSE)
cat("dataset :", manifest$dataset, "\n")
cat("source  :", manifest$source, "|", manifest$chembl_release, "| downloaded",
    manifest$download_date, "\n")
cat("licence :", manifest$licence, "\n")
cat("query   :", manifest$query, "\n\n")

cat("verification:\n")
all_ok <- TRUE
for (name in names(manifest$files)) {
  meta <- manifest$files[[name]]
  d <- sha256_of(data_path(name))
  ok <- identical(d, meta$sha256)
  all_ok <- all_ok && ok
  nrows <- length(readLines(data_path(name), warn = FALSE)) - 1L
  cat(sprintf("  %-28s %s...  recorded %s...  %-8s rows %d (recorded %d)\n",
              name, substr(d, 1, 16), substr(meta$sha256, 1, 16),
              if (ok) "MATCH" else "MISMATCH", nrows, meta$n_rows))
}
cat("\nevery file matches its recorded checksum:", all_ok, "\n")
cat("hashing the file NAME instead of its contents would give:",
    substr(digest(data_path("rock1_curation_log.csv"), algo = "sha256"), 1, 16), "... (wrong)\n")



Every file matches, and the row counts match the recorded ones as well. That single `TRUE` is the whole content of a data-availability statement: the table analysed in Units 3 to 27 of this course is provably the table described in the manifest, and anyone who downloads it can run this chunk to confirm it. Had one byte changed — a corrupted download, an editor that re-saved the file with different line endings, a colleague who "just fixed a typo" — the digest would differ and the chunk would say so. The last line is the R trap again, printed so that you recognise its signature: a plausible 64-character hex string that is a hash of the path and changes when the file is moved.

### 4.1 Why one byte is enough

The property that makes this work is the **avalanche effect**: a minimal change in the input changes about half the bits of the digest, unpredictably. The next chunk measures it on real data, by flipping a single bit in a copy of the first 100 kB of the curated table, 300 times, and measuring how many of the 256 digest bits changed each time.



In [ ]:
## The avalanche effect, measured: one byte changed, half the digest bits flip.
con <- file(data_path("rock1_chembl_curated.csv"), "rb")
block <- readBin(con, "raw", n = 100000); close(con)
base_digest <- digest(block, algo = "sha256", serialize = FALSE, raw = TRUE)

bit_distance <- function(a, b) sum(as.integer(intToBits(as.integer(xor(a, b)))))
digest_distance <- function(d1, d2)
  sum(vapply(seq_along(d1), function(i) sum(rawToBits(xor(d1[i], d2[i])) == as.raw(1)), integer(1)))

set.seed(0)
positions <- sample(length(block), 300)
distances <- vapply(positions, function(pos) {
  mutated <- block
  mutated[pos] <- xor(mutated[pos], as.raw(1))          # flip the lowest bit of one byte
  digest_distance(base_digest, digest(mutated, algo = "sha256", serialize = FALSE, raw = TRUE))
}, integer(1))

cat("digest length: 256 bits | one-bit mutations tested:", length(distances), "\n")
cat(sprintf("bits changed: min %d | mean %.1f | max %d  (expected mean 128)\n",
            min(distances), mean(distances), max(distances)))
cat("mutations producing the SAME digest:", sum(distances == 0), "\n")

p <- ggplot(data.frame(d = distances), aes(d)) +
  geom_histogram(binwidth = 2, fill = "#dce8f5", colour = PAL[["blue"]], linewidth = 0.3) +
  geom_vline(xintercept = 128, colour = PAL[["orange"]], linetype = "dashed", linewidth = 0.6) +
  annotate("text", x = 129, y = Inf, vjust = 1.5, hjust = 0, size = 2.7, colour = PAL[["orange"]],
           label = " expected: 128 of 256 bits") +
  labs(x = "digest bits changed by flipping ONE bit of a 100 kB file", y = "count",
       title = "Avalanche effect of SHA-256")
print(p)



<p class="caption"><b>Figure 5.</b> Number of digest bits that change when a single bit of a 100 kB extract of the curated table is flipped, over 300 randomly chosen positions. The distribution is centred on 128 — half of the 256 bits — exactly as a good hash function requires, and no mutation produced an unchanged digest. The practical consequence: a checksum detects a one-character edit as reliably as it detects a truncated download, so "the file is unchanged" is a statement that can be verified rather than trusted.</p>



In [ ]:
## What does verification cost? Hashing throughput on the real files, with and without streaming.
rows <- list()
for (name in c("rock1_chembl_raw.csv", "rock1_chembl_curated.csv", "screening_library.csv")) {
  path <- data_path(name)
  size_mb <- file.size(path) / 1e6
  for (algo in c("md5", "sha1", "sha256", "sha512")) {
    t0 <- proc.time()[["elapsed"]]
    digest(path, algo = algo, file = TRUE)
    dt <- proc.time()[["elapsed"]] - t0
    rows[[length(rows) + 1]] <- data.frame(file = name, MB = round(size_mb, 2), algo = algo,
                                           seconds = round(dt, 4),
                                           MB_per_s = round(size_mb / max(dt, 1e-6), 1))
  }
}
cost <- do.call(rbind, rows)
print(knitr::kable(cost, row.names = FALSE, format = "simple"))

cost$algo <- factor(cost$algo, levels = c("md5", "sha1", "sha256", "sha512"))
p <- ggplot(cost, aes(algo, MB_per_s, colour = paste0(file, " (", MB, " MB)"), group = file)) +
  geom_line(linewidth = 0.6) + geom_point(size = 2) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange", "green")])) +
  labs(x = "digest algorithm", y = "throughput (MB/s)", colour = NULL,
       title = "Checksumming is cheap, whichever algorithm you choose") +
  theme(legend.position = "bottom", legend.text = element_text(size = 7))
print(p)



<p class="caption"><b>Figure 6.</b> Hashing throughput on three real course files for four digest algorithms, measured by <code>digest(file = TRUE)</code>. Throughput is in the hundreds of megabytes per second for every algorithm, so checksumming a multi-gigabyte trajectory costs seconds, not minutes, and the stronger digests cost at most a small factor more than the weaker ones; the printed table gives the measured times. There is therefore no performance argument against verifying data, and none against choosing SHA-256 over MD5 — the only reason manifests are missing from projects is that nobody wrote them.</p>

### 4.2 Writing a manifest

A manifest is written once, by code, when the data enters `data/raw/`, and never by hand. The next chunk builds one for the three ROCK1 files from scratch and compares its field names with those of the course's own manifest — so you have both a template and a check that nothing essential is missing.



In [ ]:
## Build a manifest from scratch, in the schema this group uses.
build_manifest <- function(names, dataset, source, query, release, date, licence) {
  entries <- lapply(names, function(name) {
    path <- data_path(name)
    header <- strsplit(readLines(path, n = 1, warn = FALSE), ",")[[1]]
    list(sha256 = sha256_of(path),
         n_rows = length(readLines(path, warn = FALSE)) - 1L,
         n_cols = length(header), size_bytes = as.integer(file.size(path)))
  })
  list(dataset = dataset, source = source, query = query, chembl_release = release,
       download_date = date, licence = licence, files = setNames(entries, names),
       tooling = list(R = paste(R.version$major, R.version$minor, sep = "."),
                      data.table = as.character(packageVersion("data.table")),
                      digest = as.character(packageVersion("digest"))))
}

mine <- build_manifest(c("rock1_chembl_raw.csv", "rock1_chembl_curated.csv",
                         "rock1_curation_log.csv"),
                       dataset = "ROCK1 (CHEMBL3231) bioactivity snapshot",
                       source = "ChEMBL REST API (EBI)",
                       query = "activity.json?target_chembl_id=CHEMBL3231&standard_type__in=IC50,Ki,Kd",
                       release = "ChEMBL_37", date = "2026-10-18",
                       licence = "CC BY-SA 3.0 (ChEMBL)")
write_json(mine, file.path(SCRATCH, "manifest_mine.json"), auto_unbox = TRUE, pretty = TRUE)

cat(toJSON(mine[setdiff(names(mine), "files")], auto_unbox = TRUE, pretty = TRUE), "\n\n")
cat("per-file entries:\n")
for (n in names(mine$files)) {
  m <- mine$files[[n]]
  cat(sprintf("  %-28s rows %5d  cols %3d  sha256 %s...\n", n, m$n_rows, m$n_cols,
              substr(m$sha256, 1, 12)))
}
cat("\nchecksums identical to the course manifest:",
    all(vapply(names(mine$files), function(n)
      identical(mine$files[[n]]$sha256, manifest$files[[n]]$sha256), logical(1))), "\n")
cat("fields the course manifest has and ours lacks:",
    paste(setdiff(names(manifest), names(mine)), collapse = ", "), "\n")



The checksums we computed are identical to the recorded ones, and the comparison of field names shows what a fuller manifest adds: summary statistics and documentation hashes. Note the `tooling` field we included — the package versions that produced the files — which closes the loop with Section 3. A tool such as DVC (data version control) automates this pattern: it stores the checksum and the location in git while the bytes live elsewhere, so `git checkout` of an old commit also restores the matching data. The concept is the manifest; the tool is an implementation of it.

<div class="box warn"><h4>Hash the file, not the data frame</h4>
<p>Computing a checksum of an in-memory table — <code>digest(df)</code>, or hashing a CSV string produced by <code>fwrite()</code> to a text connection — looks equivalent and is not. <code>digest(df)</code> hashes R's <em>serialisation</em> of the object, which carries attributes, column classes, row names and the serialisation version, so it changes when nothing about the data changed and differs between R versions; a re-written CSV changes with float formatting, column order and line endings. Hash <em>files on disk</em> with <code>file = TRUE</code>, and record the number of rows and columns separately as a human-readable cross-check — exactly as the manifest above does. If you genuinely need a content hash of a table independent of its file, the defensible version is <code>digest(as.matrix(df[order(key), ]), serialize = FALSE)</code> after fixing the column order and rounding, and you must document that recipe alongside the hash.</p></div>

## 5. FAIR, applied to a QSAR dataset and to a model

The FAIR principles<sup class="c"><a href="#r21">[21]</a></sup> ask that research outputs be **F**indable (a persistent identifier, rich metadata, indexed), **A**ccessible (retrievable by that identifier over an open protocol, with metadata surviving even if the data are restricted), **I**nteroperable (standard formats and vocabularies, references to other data) and **R**eusable (a clear licence, detailed provenance, community standards). They are not a synonym for "open": a clinical dataset can be FAIR and closed, provided its metadata and access conditions are explicit<sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup>. The principles have been extended to research software<sup class="c"><a href="#r24">[24]</a></sup><sup class="c"><a href="#r25">[25]</a></sup>, which matters here because a trained QSAR model is software plus parameters plus a training set.

The next chunk scores the course dataset against sixteen concrete, checkable criteria, computing each one from the files themselves rather than from an opinion.



In [ ]:
## Score the course dataset against sixteen checkable FAIR criteria.
cur_cols <- names(load_data("rock1_chembl_curated.csv", nrows = 1))
readme <- paste(readLines(data_path("README_rock1.md"), warn = FALSE), collapse = "\n")

checks <- rbind(
  data.frame(principle = "F", criterion = "a persistent identifier (DOI) is minted for the dataset",
             satisfied = FALSE),
  data.frame(principle = "F", criterion = "the source database and release are recorded",
             satisfied = "chembl_release" %in% names(manifest)),
  data.frame(principle = "F", criterion = "the exact query is recorded",
             satisfied = nzchar(manifest$query %||% "")),
  data.frame(principle = "F", criterion = "the download date is recorded",
             satisfied = nzchar(manifest$download_date %||% "")),
  data.frame(principle = "A", criterion = "files are retrievable without special software (CSV/JSON)",
             satisfied = TRUE),
  data.frame(principle = "A", criterion = "metadata are in a machine-readable file",
             satisfied = file.exists(data_path("manifest_rock1.json"))),
  data.frame(principle = "A", criterion = "integrity is verifiable (checksums recorded)",
             satisfied = all(vapply(manifest$files, function(m) "sha256" %in% names(m), logical(1)))),
  data.frame(principle = "I", criterion = "compounds carry a standard structural identifier (InChIKey)",
             satisfied = "inchikey" %in% cur_cols),
  data.frame(principle = "I", criterion = "compounds carry database cross-references (ChEMBL ids)",
             satisfied = "chembl_ids" %in% cur_cols),
  data.frame(principle = "I", criterion = "the target is identified by a standard accession (UniProt)",
             satisfied = grepl("Q13464", readme)),
  data.frame(principle = "I", criterion = "units and scales are documented",
             satisfied = grepl("pActivity", readme) && grepl("nM", readme)),
  data.frame(principle = "R", criterion = "a licence is stated",
             satisfied = nzchar(manifest$licence %||% "")),
  data.frame(principle = "R", criterion = "a column dictionary exists",
             satisfied = lengths(regmatches(readme, gregexpr("\\|", readme))) > 40),
  data.frame(principle = "R", criterion = "the curation protocol is documented step by step",
             satisfied = grepl("curation_log|Protocol", readme)),
  data.frame(principle = "R", criterion = "provenance of derived values is recoverable (per-step log)",
             satisfied = file.exists(data_path("rock1_curation_log.csv"))),
  data.frame(principle = "R", criterion = "tool versions are recorded",
             satisfied = grepl("RDKit", readme)))

print(knitr::kable(checks, row.names = FALSE, format = "simple"))
by_letter <- aggregate(satisfied ~ principle, checks, function(x) c(sum = sum(x), size = length(x)))
by_letter <- data.frame(principle = by_letter$principle, sum = by_letter$satisfied[, "sum"],
                        size = by_letter$satisfied[, "size"])
by_letter$score_pct <- round(100 * by_letter$sum / by_letter$size)
cat("\nscore by principle:\n"); print(by_letter, row.names = FALSE)
cat(sprintf("\noverall: %d of %d criteria satisfied\n", sum(checks$satisfied), nrow(checks)))
cat("the failing criterion:", checks$criterion[!checks$satisfied], "\n")



<p class="caption"><b>Table (Section 5).</b> Sixteen FAIR criteria evaluated against the course dataset by reading its own files. Fifteen pass. The one that fails is the first: the frozen snapshot has no DOI, because it lives in a teaching repository rather than in a deposit — which is precisely the gap Section 6 closes, and precisely the gap a reviewer would flag in a submitted manuscript.</p>



In [ ]:
## Figure: the FAIR scorecard, by principle.
by_letter$label <- c("Findable", "Accessible", "Interoperable", "Reusable")[
  match(by_letter$principle, c("F", "A", "I", "R"))]
by_letter$label <- factor(by_letter$label,
                          levels = c("Findable", "Accessible", "Interoperable", "Reusable"))
p <- ggplot(by_letter, aes(label, score_pct, fill = score_pct < 100)) +
  geom_col(width = 0.55, show.legend = FALSE) +
  geom_text(aes(label = paste0(sum, "/", size)), vjust = -0.4, fontface = "bold", size = 3.1) +
  scale_fill_manual(values = c(`FALSE` = PAL[["blue"]], `TRUE` = PAL[["orange"]])) +
  scale_y_continuous(limits = c(0, 112), expand = c(0, 0)) +
  labs(x = NULL, y = "criteria satisfied (%)",
       title = "FAIR scorecard of the course's ROCK1 snapshot")
print(p)



<p class="caption"><b>Figure 7.</b> The same assessment as a scorecard. Accessible, Interoperable and Reusable are fully satisfied because the dataset ships open formats, standard identifiers (InChIKey, ChEMBL and UniProt accessions), a column dictionary, a licence and a step-by-step curation log. Findable is incomplete for one reason only — no persistent identifier — which is a five-minute fix (a Zenodo deposit) and the difference between a file and a citable research object.</p>

<div class="box key"><h4>FAIR for a trained model</h4>
<p>A model is FAIR when: the <em>training set</em> is identified by a DOI and a checksum; the <em>code</em> that trained it is in a tagged, archived repository; the <em>hyperparameters and seed</em> are recorded; the <em>metrics</em> are reported with their validation protocol (which split, how many repetitions); the <em>applicability domain</em> is stated (Unit 17); and the serialised object carries a licence and a model card (Unit 26). Note that an <code>.rds</code> file of a fitted R model is <em>not</em> interoperable: it is R's serialisation format, it can silently depend on the package that defined the model class, and a <code>lm</code> object in particular stores the whole training frame inside itself — so saving one both bloats the file and may leak data you did not intend to publish. This group therefore also stores the training script and the data checksum, never only the binary.</p></div>

## 6. Licences, identifiers and the three files every release carries

Three small files turn a folder into a citable research object, and all three are written once.

- **`LICENSE`** states what others may legally do. For **code**, permissive licences (MIT, BSD-3-Clause, Apache-2.0) allow reuse including in closed software and are the group's default; copyleft licences (GPL) require derived work to carry the same licence. R packages themselves are predominantly GPL-licensed, which matters if you *bundle* package source into a release, though merely depending on a GPL package does not impose GPL on your analysis scripts. For **data**, CC0 (public domain dedication) and CC BY (attribution) are the standard choices. The decisive constraint for this course: ChEMBL data are distributed under **CC BY-SA 3.0**, a *share-alike* licence, so a table derived from ChEMBL must be redistributed under the same terms — you may not re-licence the course's ROCK1 snapshot as CC0, and a dataset that mixes a share-alike source with an incompatible one cannot be redistributed at all. Check the licence of every source *before* you build a derived table, not after.
- **`CITATION.cff`** is a machine-readable citation file (YAML) that GitHub and Zenodo read to generate the correct citation. It is the software-citation principles<sup class="c"><a href="#r26">[26]</a></sup> made executable. R has its own, older mechanism — a `CITATION` file in `inst/`, read by `citation("pkg")` — and the two coexist; write both if you release a package.
- **`README.md`** is the human entry point: the question, the layout, the exact commands that reproduce the result, and the data-availability statement. Guidance on documenting scientific software applies in full<sup class="c"><a href="#r27">[27]</a></sup>.

The **persistent identifier** comes from a deposit. Connecting a GitHub repository to Zenodo and creating a release mints a DOI for that exact snapshot of the code, plus a concept DOI that always resolves to the latest version. Data deposits work the same way. A DOI is not a convenience: a repository URL can be renamed or deleted, while a DOI resolves permanently and is what a journal's data-availability statement should contain<sup class="c"><a href="#r28">[28]</a></sup><sup class="c"><a href="#r29">[29]</a></sup>.



In [ ]:
## Write the three release files, then parse CITATION.cff back to prove it is machine-readable.
release <- file.path(SCRATCH, "release")
dir.create(release, showWarnings = FALSE)

writeLines(c("MIT License", "",
  "Copyright (c) 2026 Laboratorio de Bioinformatica y Quimica Computacional (LBQC),",
  "Universidad Catolica del Maule", "",
  "Permission is hereby granted, free of charge, to any person obtaining a copy",
  "of this software and associated documentation files (the \"Software\"), to deal",
  "in the Software without restriction..."), file.path(release, "LICENSE"))

citation_cff <- list(
  "cff-version" = "1.2.0",
  message = "If you use this dataset or code, please cite the following.",
  title = "ROCK1 bioactivity snapshot and analysis code (LBQC data-science course)",
  authors = list(list("family-names" = "Bustos", "given-names" = "Daniel",
                      affiliation = "Universidad Catolica del Maule")),
  version = "1.0.0", "date-released" = "2026-10-18", license = "CC-BY-SA-3.0",
  type = "dataset", keywords = list("ROCK1", "ChEMBL", "QSAR", "reproducibility"),
  references = list(list(type = "article", title = "The ChEMBL database in 2023",
                         doi = "10.1093/nar/gkad1004", year = 2024,
                         authors = list(list("family-names" = "Zdrazil",
                                             "given-names" = "Barbara")))))
write_yaml(citation_cff, file.path(release, "CITATION.cff"))

writeLines(c("# ROCK1 bioactivity snapshot", "", "## Data availability",
  sprintf("ChEMBL release %s, query `%s`,", manifest$chembl_release, manifest$query),
  sprintf("downloaded %s; SHA-256 of each file in `manifest_rock1.json`.", manifest$download_date),
  sprintf("Licence: %s.", manifest$licence), "", "## Reproduce", "```",
  "R -q -e 'renv::restore()'", "Rscript R/build_dataset.R", "```"),
  file.path(release, "README.md"))

parsed <- read_yaml(file.path(release, "CITATION.cff"))
cat("files written:", paste(sort(list.files(release)), collapse = ", "), "\n\n")
cat("CITATION.cff parses back to a list with keys:", paste(sort(names(parsed)), collapse = ", "), "\n")
cat("citation title :", parsed$title, "\n")
cat("licence        :", parsed$license, "| version:", parsed$version,
    "| released:", parsed$`date-released`, "\n")
cat("cited reference:", parsed$references[[1]]$title, "doi:", parsed$references[[1]]$doi, "\n")
cat("\nlicence compatibility check: dataset licence is", manifest$licence, "->",
    if (grepl("SA", manifest$licence)) "share-alike: derived tables must keep CC BY-SA"
    else "no share-alike obligation", "\n")



The citation file parses back into a list, which is the only test that matters — a `CITATION.cff` with a YAML error is silently ignored by GitHub and by Zenodo. Note the fields that earn their place: a `version` and a `date-released` (so a reader can cite the exact snapshot), a `license` identifier from the SPDX list rather than free text, and a `references` block pointing at the source database's own paper, which is how attribution chains are kept intact.



In [ ]:
## Conceptual figure: choosing a licence for code and for data.
boxes <- data.frame(
  x = c(2.8, 1.3, 4.3, 0.3, 2.3, 3.5, 5.6),
  y = c(2.6, 1.5, 1.5, 0.4, 0.4, 0.4, 0.4),
  w = c(2.6, 1.4, 1.4, 2.1, 2.1, 2.1, 2.3),
  label = c("What are you releasing?", "CODE", "DATA",
            "reuse anywhere\n-> MIT / BSD-3 / Apache-2.0",
            "derivatives must stay open\n-> GPL-3.0",
            "no source restriction\n-> CC0 or CC BY-4.0",
            "built from CC BY-SA source\n(e.g. ChEMBL)\n-> must stay CC BY-SA"),
  fill = c("#f0f0f0", "#dce8f5", "#f6dcdc", "#dce8f5", "#dce8f5", "#f6dcdc", "#f6dcdc"),
  bold = c(TRUE, TRUE, TRUE, FALSE, FALSE, FALSE, FALSE), stringsAsFactors = FALSE)
arrows <- data.frame(x = c(2.8, 2.8, 1.3, 1.3, 4.3, 4.3), y = rep(c(2.42, 1.32), each = 3)[c(1,1,2,2,2,2)],
                     xend = c(1.3, 4.3, 0.3, 2.3, 3.5, 5.6), yend = c(1.68, 1.68, rep(0.58, 4)))

p <- ggplot() +
  geom_tile(data = boxes, aes(x, y, width = w, height = 0.56, fill = I(fill)),
            colour = "grey35", linewidth = 0.4) +
  geom_text(data = boxes, aes(x, y, label = label,
                              fontface = ifelse(bold, "bold", "plain")), size = 2.4) +
  geom_segment(data = arrows, aes(x = x, y = y, xend = xend, yend = yend), colour = "grey55",
               arrow = arrow(length = unit(0.07, "cm"), type = "closed")) +
  annotate("text", x = 2.9, y = -0.1, size = 2.7, colour = PAL[["orange"]],
           label = "and always: a DOI from a deposit, plus CITATION.cff") +
  scale_x_continuous(limits = c(-0.9, 6.9)) + scale_y_continuous(limits = c(-0.3, 3.1)) +
  labs(title = "Licence decisions, and the obligation a share-alike source imposes") +
  theme_void(base_size = 9.5) + theme(plot.title = element_text(face = "bold", size = 10.5))
print(p)



<p class="caption"><b>Figure 8.</b> The licence decision as this group makes it. The right-hand branch is the one that catches students: a derived table inherits the obligations of its source, so a snapshot built from ChEMBL (CC BY-SA 3.0) must be redistributed under share-alike terms even if your own code is MIT. Code and data are licensed separately and may have different licences in the same repository — the <code>LICENSE</code> file covers the code, and the data licence is stated in the manifest and the README.</p>

## 7. A reproducibility audit

The final exercise of this unit is the one a reviewer performs. We create a project that looks plausible and contains six deliberate, extremely common defects, then audit it with code. Every check below is mechanical: it reads files and answers yes or no, which is what makes it usable as a pre-submission checklist.



In [ ]:
## Build a plausible-looking project with six deliberate defects, then audit it.
bad <- file.path(SCRATCH, "plausible_project")
for (d in c("data/raw", "analysis", "R", "figures"))
  dir.create(file.path(bad, d), recursive = TRUE, showWarnings = FALSE)
writeLines(c("id,value", "1,25"), file.path(bad, "data/raw/activities.csv"))
writeLines(c("```{r}",
             "df <- read.csv('/home/student/Desktop/final_data_v3_FINAL.csv')",
             "test_i <- sample(nrow(df), 0.25 * nrow(df))",
             "print(summary(fit)$r.squared)", "```"),
           file.path(bad, "analysis/analysis.Rmd"))
writeLines(c("fit <- function(X, y) {", "  idx <- sample(length(y))", "  idx", "}"),
           file.path(bad, "R/model.R"))
writeLines(c("# Project", "", "Run the notebook."), file.path(bad, "README.md"))

## The pattern below is assembled from pieces so that this teaching page itself
## contains no absolute path - the audit it implements would otherwise flag its own source.
ABS_PATH_RX <- paste0("['\"](/", "Users/|/home/|C:\\\\)")

audit <- function(root) {
  code_files <- list.files(root, pattern = "\\.(R|Rmd)$", recursive = TRUE, full.names = TRUE)
  code_text <- paste(unlist(lapply(code_files, readLines, warn = FALSE)), collapse = "\n")
  readme <- if (file.exists(file.path(root, "README.md")))
    paste(readLines(file.path(root, "README.md"), warn = FALSE), collapse = "\n") else ""
  data.frame(
    check = c("environment file present", "data manifest with checksums",
              "data kept out of version control", "no absolute path in code",
              "randomness is seeded", "README states how to reproduce",
              "LICENSE present", "CITATION.cff present"),
    passes = c(
      any(file.exists(file.path(root, c("renv.lock", "DESCRIPTION")))),
      length(list.files(root, pattern = "^manifest.*json$", recursive = TRUE)) > 0,
      file.exists(file.path(root, ".gitignore")),
      !grepl(ABS_PATH_RX, code_text),
      grepl("set\\.seed", code_text),
      grepl("renv::restore|Rscript R/", readme),
      file.exists(file.path(root, "LICENSE")),
      file.exists(file.path(root, "CITATION.cff"))))
}

report <- audit(bad)
cat("audit of the plausible-looking project:\n")
print(knitr::kable(report, row.names = FALSE, format = "simple"))
cat(sprintf("\npassed %d of %d checks\n", sum(report$passes), nrow(report)))
report_good <- audit(release)
cat(sprintf("the same audit on the release folder of Section 6: passed %d of %d checks\n",
            sum(report_good$passes), nrow(report_good)))



The plausible project fails every one of the eight checks, and each failure corresponds to a sentence a reviewer writes. The absolute path `/home/student/Desktop/final_data_v3_FINAL.csv` means nobody else can run the document and nobody knows which of the versions implied by that file name was used. The `sample()` call with no `set.seed()` before it — in the analysis *and* inside `R/model.R` — means the reported \( R^2 \) is one draw from the histogram of Figure 4. The missing lock file means the result cannot be reproduced even with the code. The missing manifest means the data are unidentified. The missing `.gitignore` means a committed CSV that is still unverified. The `README` that says "run the notebook" documents nothing.

Note that the release folder written in Section 6 does not pass everything either — it has a `LICENSE` and a `CITATION.cff` but no lock file or manifest of its own, because it is only a fragment. A real project passes all eight; that is the bar.

<div class="exercise"><h4>Exercise 2</h4>
<p>Fix the plausible project in code: write a <code>DESCRIPTION</code> and a <code>.gitignore</code> into it, build a manifest for <code>data/raw/activities.csv</code> with <code>build_manifest()</code>-style logic, rewrite the analysis source to use a relative path and an explicit <code>set.seed(0)</code>, and re-run <code>audit()</code>. How many checks pass now? Then add a ninth check of your own — for example, that every figure in <code>figures/</code> has a script in <code>R/</code> that produces it, or that no <code>.RData</code> file is present — and argue in two sentences why that check matters.</p></div>



In [ ]:
## Figure: the two audits side by side.
both <- rbind(data.frame(check = report$check, passes = as.integer(report$passes),
                         project = "plausible-looking project"),
              data.frame(check = report_good$check, passes = as.integer(report_good$passes),
                         project = "Section 6 release folder"))
both$check <- factor(both$check, levels = rev(report$check))
p <- ggplot(both, aes(passes, check, fill = project)) +
  geom_col(position = position_dodge(width = 0.75), width = 0.7) +
  scale_fill_manual(values = unname(PAL[c("orange", "blue")])) +
  scale_x_continuous(breaks = c(0, 1), labels = c("fails", "passes"), limits = c(0, 1.05)) +
  labs(x = NULL, y = NULL, fill = NULL, title = "A reproducibility audit is mechanical") +
  theme(legend.position = "bottom", axis.text.y = element_text(size = 7.5))
print(p)



<p class="caption"><b>Figure 9.</b> The eight audit checks applied to the deliberately defective project (orange) and to the release folder of Section 6 (blue). Neither is complete, and that is the teaching point: reproducibility is not a property a project has or lacks in general, but a list of specific, checkable facts, each of which is cheap to fix once it has been named. Run this audit on your own project before your supervisor does.</p>



In [ ]:
## Clean up every temporary artefact created by this document.
unlink(SCRATCH, recursive = TRUE, force = TRUE)
cat("scratch directory removed:", !dir.exists(SCRATCH), "\n")
cat("objects in the miniature git store (in memory only):", length(ls(OBJECTS)), "\n")
cat("the frozen tables in ../data/ were read, never modified:",
    file.exists(data_path("manifest_rock1.json")), "\n")



## 8. Summary

You should now be able to explain, without notes:

- The difference between repeatable, reproducible and replicable, and which of the three this unit buys.
- Git's object model: blob, tree, commit, tag; content addressing; a branch as a movable pointer; the history as a DAG.
- How an object identifier is computed, why one changed byte changes the whole identifier, and why `digest()` needs `serialize = FALSE` to hash bytes and `file = TRUE` to hash a file.
- What a merge base is, why most merges are automatic, what a conflict file contains, and how to resolve one.
- Which files must never enter a repository, where data and secrets go instead, and why knitted output and `.RData` are unmergeable.
- The three levels of environment specification in R — `DESCRIPTION`, `sessionInfo()`, `renv.lock` — what each is for, and what none of them captures.
- Why an unseeded split makes a reported metric one draw from a distribution, roughly how wide that distribution was for the model in Section 3.3, and why R's global stream forces a seed next to every sampling call.
- What a data manifest records, how to verify a file against it, and why hashing a data frame is not the same as hashing a file.
- The FAIR principles applied to a dataset and to a trained model, the one criterion the course dataset currently fails, and why an `.rds` of a fitted model is not an interoperable deliverable.
- How to choose a licence for code and for data, what a share-alike source obliges you to do, and what `CITATION.cff` and a DOI add.
- The eight mechanical checks of a reproducibility audit.

<div class="box bridge"><h4>Bridge — where this continues</h4>
<p><a href="../U3_Getting_Data/index.html">Unit 3</a> fills <code>data/raw/</code> from APIs, databases and SQL, and every download it makes is the thing this unit's manifest describes; <a href="../U1_Toolkit/index.html">Unit 1</a> built the project folder this unit put under version control. The later units of this R edition follow the sequence of the <a href="../../Python/index.html">Python edition</a>, whose counterparts already exist: <a href="../U6_Data_Quality/index.html">data quality</a> turns the curation protocol documented here into a reproducible, logged pipeline; <a href="../U12_Validation/index.html">validation</a> replaces the single split of Section 3.3 with a protocol that reports its own dispersion; <a href="../U23_Communication/index.html">scientific communication</a> writes the data-availability statement; <a href="../U24_Software_Engineering/index.html">software engineering</a> turns <code>R/</code> into a tested package with continuous integration, and <a href="../U25_Scaling_HPC/index.html">scaling up</a> puts the pinned environment into a container. In the sister courses, the cheminformatics elective applies the same provenance discipline to <a href="../../../Cheminformatics/U6_Databases/index.html">chemical databases</a> and <a href="../../../Cheminformatics/U7_Data_Curation/index.html">curation</a>, BGM324 shows the simulation outputs that need it in <a href="../../../BGM324/05_Molecular_Dynamics/index.html">molecular dynamics</a>, and the R edition of BGM411 (<a href="../../../BGM411/R/index.html">study material</a>) uses the same seeding and reporting rules for model code.</p></div>

## 9. Self-assessment

<details><summary>Q1. Two identical files sit in two unrelated repositories on two continents. What can you say about their git object identifiers?</summary>
<p>They are identical, because the identifier is the SHA-1 of <code>"blob " + length + NUL + content</code> — a function of content alone, with no contribution from the file name, path, timestamp or repository. This is what lets git deduplicate storage and verify integrity with the same mechanism.</p></details>

<details><summary>Q2. Why does committing an unchanged file ten times not store ten copies?</summary>
<p>Because each commit stores a tree that <em>points at</em> blob ids. If the content has not changed, the blob id has not changed, so every tree references the same existing object. Only the small tree and commit objects are new. This is why frequent commits are cheap and why "I will commit when it is finished" is bad advice.</p></details>

<details><summary>Q3. You compute <code>digest("data/raw/activities.csv", algo = "sha256")</code> and get a 64-character hex string. Is that the checksum of the file?</summary>
<p>No. Without <code>file = TRUE</code>, <code>digest()</code> hashed the <em>character string</em> containing the path, so the answer changes if the file is moved and does not change if the file's contents change — a confident, meaningless number. Two arguments matter: <code>file = TRUE</code> to hash the bytes of a file, and <code>serialize = FALSE</code> to hash the bytes of a raw vector rather than R's serialisation of it. Both were demonstrated in Sections 2.1 and 4.</p></details>

<details><summary>Q4. A merge produced a conflict in one line of a file and merged forty other changed lines silently. Explain both behaviours in terms of the merge base.</summary>
<p>Git compares each side against the most recent common ancestor. Where only one side differs from the base, that side's version is taken (forty lines). Where both sides differ from the base and from each other, no rule can decide, so the two versions are written into the file between markers for a human to resolve (one line).</p></details>

<details><summary>Q5. Your collaborator asks you to commit a 400 MB trajectory "so we both have it". What do you answer?</summary>
<p>That git stores every version forever and in every clone, so the repository would grow by 400 MB per version for everyone, permanently. The trajectory goes to a data deposit (with a DOI) or an institutional store; the repository carries its checksum, its location and the script that downloads it. If it must be versioned with the code, git-lfs or a data-version-control tool keeps pointers in git and bytes elsewhere.</p></details>

<details><summary>Q6. You commit <code>renv.lock</code> with <code>dplyr</code> pinned to an exact version. Is your analysis now reproducible?</summary>
<p>More reproducible, not reproducible. The lock file pins CRAN packages and their sources, but not the R version itself (record it separately, as the lock file's <code>R</code> block does), not the compiler, not the system libraries behind packages such as <code>arrow</code>, and not the BLAS and LAPACK implementations behind every matrix operation — which <code>sessionInfo()</code> prints and which differ between a Linux cluster and a laptop. A container pins the whole user-space stack. Numerical results can still differ in the last digits across architectures.</p></details>

<details><summary>Q7. A manuscript reports a single test \( R^2 \) for this descriptor model, with no seed and no repetitions. What is the honest objection, quantitatively?</summary>
<p>That the number is one draw from the distribution of Figure 4: refitting the same model on the same data with only the split changed produced a spread between the unluckiest and luckiest seed that is a large fraction of the value itself. Without repetitions one cannot tell a real improvement from a lucky split. Report a median and a dispersion over repeated, seeded splits.</p></details>

<details><summary>Q8. Why hash a file on disk rather than the data frame you loaded from it?</summary>
<p>Because the digest must identify the bytes a collaborator can download. <code>digest(df)</code> hashes R's serialisation of the object, which carries column classes, attributes and row names and changes between R versions; a table re-written with <code>fwrite()</code> depends on float formatting, column order and line endings. Hash the file with <code>file = TRUE</code> and record rows and columns separately as a readable cross-check.</p></details>

<details><summary>Q9. A dataset has a DOI, a licence and rich metadata, but access requires a signed agreement. Is it FAIR?</summary>
<p>It can be. FAIR requires that the metadata be findable and accessible and that the access conditions be explicit and machine-readable; it does not require the data themselves to be open. "FAIR" and "open" are different axes, and conflating them is a common error in data-management plans.</p></details>

<details><summary>Q10. Your code is MIT-licensed and your derived table was built from ChEMBL. May you release the table as CC0?</summary>
<p>No. ChEMBL is distributed under CC BY-SA 3.0, a share-alike licence, so a derived table must be redistributed under the same terms; the MIT licence on your code is unaffected and the two coexist in one repository. Check source licences before building a derived table, especially before combining sources whose licences may be incompatible.</p></details>

<details><summary>Q11. What does a Zenodo DOI add that a GitHub URL does not?</summary>
<p>Permanence and specificity. A repository can be renamed, made private or deleted, and its URL then resolves to nothing; a DOI resolves permanently to an archived snapshot. A Zenodo release also mints a version DOI (this exact snapshot) alongside a concept DOI (always the latest), which is what a data-availability statement and a <code>CITATION.cff</code> should carry.</p></details>

<details><summary>Q12. Why does this unit compute git's hashes instead of simply teaching the commands?</summary>
<p>Because the commands are unmemorable until the model is clear, and because the model explains their behaviour: why identifiers are long hexadecimal strings, why commits are cheap, why history is a graph, why a conflict is a refusal rather than a failure, and why <code>--force</code> is dangerous. Once content addressing is understood, the command list in Section 2.4 reads as a description of simple graph operations — and the same mechanism, applied with SHA-256 to files instead of SHA-1 to blobs, is the manifest of Section 4.</p></details>

## Glossary

<dl class="gloss">
<dt>Avalanche effect</dt><dd>The property that flipping one input bit changes about half the output bits of a hash function.</dd>
<dt>Blob / tree / commit</dt><dd>Git's object types: file contents, a directory listing, and a snapshot with parents, author and message.</dd>
<dt>Branch</dt><dd>A movable pointer to a commit; creating one is nearly free.</dd>
<dt>CITATION.cff</dt><dd>A machine-readable YAML citation file read by GitHub and Zenodo; R packages additionally carry an <code>inst/CITATION</code> file read by <code>citation()</code>.</dd>
<dt>Content addressing</dt><dd>Naming an object by the hash of its contents, so the name proves the contents.</dd>
<dt>DAG</dt><dd>Directed acyclic graph; the shape of a git history, in which a merge commit has two parents.</dd>
<dt>DOI</dt><dd>Digital object identifier; a permanently resolvable name for a deposited snapshot.</dd>
<dt>FAIR</dt><dd>Findable, Accessible, Interoperable, Reusable — principles for research outputs; not a synonym for open.</dd>
<dt>Lock file</dt><dd>A record of exact package versions and their sources from which an installation is reproducible; <code>renv.lock</code> in R.</dd>
<dt>Manifest</dt><dd>A text record of each data file's source, query, date, licence and checksum.</dd>
<dt>Merge base</dt><dd>The most recent common ancestor of two branches; the reference against which both sides are compared.</dd>
<dt>Merge conflict</dt><dd>Git's refusal to choose between two edits of the same region; resolved by a human, not an error.</dd>
<dt>Provenance</dt><dd>The documented chain from an original source to the file in front of you.</dd>
<dt>Pull request</dt><dd>A request to merge a branch, carrying review and discussion; the unit of collaboration on GitHub.</dd>
<dt>renv</dt><dd>The R package that gives a project its own library and records exact versions in <code>renv.lock</code>.</dd>
<dt>Serialisation</dt><dd>R's binary encoding of an object (<code>saveRDS</code>, and what <code>digest()</code> hashes by default); version-dependent, which is why checksums are taken of files.</dd>
<dt>sessionInfo()</dt><dd>R's report of the interpreter, platform, BLAS/LAPACK libraries and attached packages at the moment it is called.</dd>
<dt>SHA-1 / SHA-256</dt><dd>Cryptographic hash functions producing 160-bit and 256-bit digests; git names objects with the former, integrity records use the latter.</dd>
<dt>Share-alike</dt><dd>A licence condition (as in CC BY-SA) requiring derived works to carry the same licence.</dd>
<dt>Staging area</dt><dd>The set of blobs that <code>git commit</code> will turn into the next tree.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Eglen SJ, Marwick B, Halchenko YO, et al. Toward standard practices for sharing computer code and programs in neuroscience. <em>Nat Neurosci</em>. 2017;20(6):770-773. <a href="https://doi.org/10.1038/nn.4550">doi:10.1038/nn.4550</a></li>
<li id="r2">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL database in 2023: a drug discovery platform spanning multiple bioactivity data types and time periods. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r3">Mendez D, Gaulton A, Bento AP, et al. ChEMBL: towards direct deposition of bioassay data. <em>Nucleic Acids Res</em>. 2019;47(D1):D930-D940. <a href="https://doi.org/10.1093/nar/gky1075">doi:10.1093/nar/gky1075</a></li>
<li id="r4">Walters WP. Modeling, informatics, and the quest for reproducibility. <em>J Chem Inf Model</em>. 2013;53(7):1529-1530. <a href="https://doi.org/10.1021/ci400197w">doi:10.1021/ci400197w</a></li>
<li id="r5">Baker M. 1,500 scientists lift the lid on reproducibility. <em>Nature</em>. 2016;533(7604):452-454. <a href="https://doi.org/10.1038/533452a">doi:10.1038/533452a</a></li>
<li id="r6">Stodden V, Seiler J, Ma Z. An empirical analysis of journal policy effectiveness for computational reproducibility. <em>Proc Natl Acad Sci USA</em>. 2018;115(11):2584-2589. <a href="https://doi.org/10.1073/pnas.1708290115">doi:10.1073/pnas.1708290115</a></li>
<li id="r7">Hutson M. Artificial intelligence faces reproducibility crisis. <em>Science</em>. 2018;359(6377):725-726. <a href="https://doi.org/10.1126/science.359.6377.725">doi:10.1126/science.359.6377.725</a></li>
<li id="r8">Gundersen OE, Kjensmo S. State of the art: reproducibility in artificial intelligence. <em>Proc AAAI Conf Artif Intell</em>. 2018;32(1):1644-1651. <a href="https://doi.org/10.1609/aaai.v32i1.11503">doi:10.1609/aaai.v32i1.11503</a></li>
<li id="r9">Haibe-Kains B, Adam GA, Hosny A, et al. Transparency and reproducibility in artificial intelligence. <em>Nature</em>. 2020;586(7829):E14-E16. <a href="https://doi.org/10.1038/s41586-020-2766-y">doi:10.1038/s41586-020-2766-y</a></li>
<li id="r10">Sandve GK, Nekrutenko A, Taylor J, Hovig E. Ten simple rules for reproducible computational research. <em>PLoS Comput Biol</em>. 2013;9(10):e1003285. <a href="https://doi.org/10.1371/journal.pcbi.1003285">doi:10.1371/journal.pcbi.1003285</a></li>
<li id="r11">Heil BJ, Hoffman MM, Markowetz F, Lee SI, Greene CS, Hicks SC. Reproducibility standards for machine learning in the life sciences. <em>Nat Methods</em>. 2021;18(10):1132-1135. <a href="https://doi.org/10.1038/s41592-021-01256-7">doi:10.1038/s41592-021-01256-7</a></li>
<li id="r12">Wilson G, Bryan J, Cranston K, Kitzes J, Nederbragt L, Teal TK. Good enough practices in scientific computing. <em>PLoS Comput Biol</em>. 2017;13(6):e1005510. <a href="https://doi.org/10.1371/journal.pcbi.1005510">doi:10.1371/journal.pcbi.1005510</a></li>
<li id="r13">Peng RD. Reproducible research in computational science. <em>Science</em>. 2011;334(6060):1226-1227. <a href="https://doi.org/10.1126/science.1213847">doi:10.1126/science.1213847</a></li>
<li id="r14">Perez-Riverol Y, Gatto L, Wang R, et al. Ten simple rules for taking advantage of git and GitHub. <em>PLoS Comput Biol</em>. 2016;12(7):e1004947. <a href="https://doi.org/10.1371/journal.pcbi.1004947">doi:10.1371/journal.pcbi.1004947</a></li>
<li id="r15">Blischak JD, Davenport ER, Wilson G. A quick introduction to version control with Git and GitHub. <em>PLoS Comput Biol</em>. 2016;12(1):e1004668. <a href="https://doi.org/10.1371/journal.pcbi.1004668">doi:10.1371/journal.pcbi.1004668</a></li>
<li id="r16">Ram K. Git can facilitate greater reproducibility and increased transparency in science. <em>Source Code Biol Med</em>. 2013;8(1):7. <a href="https://doi.org/10.1186/1751-0473-8-7">doi:10.1186/1751-0473-8-7</a></li>
<li id="r17">Boettiger C. An introduction to Docker for reproducible research. <em>ACM SIGOPS Oper Syst Rev</em>. 2015;49(1):71-79. <a href="https://doi.org/10.1145/2723872.2723882">doi:10.1145/2723872.2723882</a></li>
<li id="r18">Grüning B, Chilton J, Köster J, et al. Practical computational reproducibility in the life sciences. <em>Cell Syst</em>. 2018;6(6):631-635. <a href="https://doi.org/10.1016/j.cels.2018.03.014">doi:10.1016/j.cels.2018.03.014</a></li>
<li id="r19">Nüst D, Sochat V, Marwick B, et al. Ten simple rules for writing Dockerfiles for reproducible data science. <em>PLoS Comput Biol</em>. 2020;16(11):e1008316. <a href="https://doi.org/10.1371/journal.pcbi.1008316">doi:10.1371/journal.pcbi.1008316</a></li>
<li id="r20">Artrith N, Butler KT, Coudert FX, et al. Best practices in machine learning for chemistry. <em>Nat Chem</em>. 2021;13(6):505-508. <a href="https://doi.org/10.1038/s41557-021-00716-z">doi:10.1038/s41557-021-00716-z</a></li>
<li id="r21">Wilkinson MD, Dumontier M, Aalbersberg IJ, et al. The FAIR guiding principles for scientific data management and stewardship. <em>Sci Data</em>. 2016;3:160018. <a href="https://doi.org/10.1038/sdata.2016.18">doi:10.1038/sdata.2016.18</a></li>
<li id="r22">Mons B, Neylon C, Velterop J, Dumontier M, da Silva Santos LOB, Wilkinson MD. Cloudy, increasingly FAIR; revisiting the FAIR data guiding principles for the European Open Science Cloud. <em>Inf Serv Use</em>. 2017;37(1):49-56. <a href="https://doi.org/10.3233/ISU-170824">doi:10.3233/ISU-170824</a></li>
<li id="r23">Jacobsen A, de Miranda Azevedo R, Juty N, et al. FAIR principles: interpretations and implementation considerations. <em>Data Intell</em>. 2020;2(1-2):10-29. <a href="https://doi.org/10.1162/dint_r_00024">doi:10.1162/dint_r_00024</a></li>
<li id="r24">Lamprecht AL, Garcia L, Kuzak M, et al. Towards FAIR principles for research software. <em>Data Sci</em>. 2019;3(1):37-59. <a href="https://doi.org/10.3233/DS-190026">doi:10.3233/DS-190026</a></li>
<li id="r25">Barker M, Chue Hong NP, Katz DS, et al. Introducing the FAIR principles for research software. <em>Sci Data</em>. 2022;9(1):622. <a href="https://doi.org/10.1038/s41597-022-01710-x">doi:10.1038/s41597-022-01710-x</a></li>
<li id="r26">Smith AM, Katz DS, Niemeyer KE. Software citation principles. <em>PeerJ Comput Sci</em>. 2016;2:e86. <a href="https://doi.org/10.7717/peerj-cs.86">doi:10.7717/peerj-cs.86</a></li>
<li id="r27">Lee BD. Ten simple rules for documenting scientific software. <em>PLoS Comput Biol</em>. 2018;14(12):e1006561. <a href="https://doi.org/10.1371/journal.pcbi.1006561">doi:10.1371/journal.pcbi.1006561</a></li>
<li id="r28">Nosek BA, Alter G, Banks GC, et al. Promoting an open research culture. <em>Science</em>. 2015;348(6242):1422-1425. <a href="https://doi.org/10.1126/science.aab2374">doi:10.1126/science.aab2374</a></li>
<li id="r29">Chen X, Dallmeier-Tiessen S, Dasler R, et al. Open is not enough. <em>Nat Phys</em>. 2018;15(2):113-119. <a href="https://doi.org/10.1038/s41567-018-0342-2">doi:10.1038/s41567-018-0342-2</a></li>
<li id="r30">Munafò MR, Nosek BA, Bishop DVM, et al. A manifesto for reproducible science. <em>Nat Hum Behav</em>. 2017;1:0021. <a href="https://doi.org/10.1038/s41562-016-0021">doi:10.1038/s41562-016-0021</a></li>
<li id="r31">Kapoor S, Narayanan A. Leakage and the reproducibility crisis in machine-learning-based science. <em>Patterns</em>. 2023;4(9):100804. <a href="https://doi.org/10.1016/j.patter.2023.100804">doi:10.1016/j.patter.2023.100804</a></li>
<li id="r32">Wilson G, Aruliah DA, Brown CT, et al. Best practices for scientific computing. <em>PLoS Biol</em>. 2014;12(1):e1001745. <a href="https://doi.org/10.1371/journal.pbio.1001745">doi:10.1371/journal.pbio.1001745</a></li>
<li id="r33">Community TTW. The Turing Way: a handbook for reproducible, ethical and collaborative research. <em>Zenodo</em>. 2022. <a href="https://doi.org/10.5281/zenodo.3233853">doi:10.5281/zenodo.3233853</a></li>
<li id="r34">Papadatos G, Gaulton A, Hersey A, Overington JP. Activity, assay and target data curation and quality in the ChEMBL database. <em>J Comput Aided Mol Des</em>. 2015;29(9):885-896. <a href="https://doi.org/10.1007/s10822-015-9860-5">doi:10.1007/s10822-015-9860-5</a></li>
<li id="r35">Gentleman R, Temple Lang D. Statistical analyses and reproducible research. <em>J Comput Graph Stat</em>. 2007;16(1):1-23. <a href="https://doi.org/10.1198/106186007X178663">doi:10.1198/106186007X178663</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Chacon S, Straub B. <em>Pro Git</em>. 2nd ed. Apress; 2014. (Free online; chapters 1–3 and 10 — "Git Internals" — are the companion reading for Section 2.)</li>
<li>Bryan J, Hester J. <em>Happy Git and GitHub for the useR</em>. (Free online; the practical companion for using git from RStudio and the shell.)</li>
<li>Xie Y, Allaire JJ, Grolemund G. <em>R Markdown: The Definitive Guide</em>. Chapman and Hall/CRC; 2018. (Free online.)</li>
<li>Wilson G et al. <em>Research Software Engineering with Python</em>. CRC Press; 2021. (Language-agnostic in everything that matters here.)</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><a href="https://book.the-turing-way.org/">The Turing Way</a> — "Guide for reproducible research", the most complete free treatment of this unit's material.</li>
<li><a href="https://software-carpentry.org/lessons/">Software Carpentry</a> — the "Version Control with Git" lesson, two hours, hands-on.</li>
<li><a href="https://rstudio.github.io/renv/articles/renv.html">renv: Project Environments</a> — the lock-file workflow of Section 3.</li>
<li><a href="https://citation-file-format.github.io/">Citation File Format</a> — the <code>CITATION.cff</code> schema and an online generator.</li>
<li><a href="https://choosealicense.com/">choosealicense.com</a> and <a href="https://spdx.org/licenses/">SPDX licence list</a> — licence texts and identifiers.</li>
<li><a href="https://zenodo.org/">Zenodo</a> — DOI minting for GitHub releases and datasets; see its GitHub integration documentation.</li>
</ul>
